# Lending Model — score the new 100,000 customers (eligibility + default risk)\n\nThis is the production version of the two lending notebooks built earlier, now targeting the new 100,000 customers and writing straight into `FactCustomerLending`. It's a **two-stage pipeline**, mirroring real underwriting order:\n\n1. **Stage 1 — Eligibility.** Train on the seed 100,000 to predict `LendingModelScore` from base profile features. Derive `Eligible` (score > 0.55) and generate a plausible `ApprovedLimit` tied to account balance (the original data ties this to `CustomerId` alone, so there's nothing real to model here — it's generated, not predicted, same as `AccountNumber`/`CIF` elsewhere).\n2. **Stage 2 — Default risk.** Train a second model on the seed 100,000 to predict `DefaultRiskScore` — this time using the base profile features **plus the Stage 1 outputs** (`LendingModelScore`, `Eligible`, `ApprovedLimit`) as additional legitimate inputs, since a lender's own approval signals are real evidence about default risk, not circular with it. Derive `Defaulted` (score > 0.88).\n\nSame caveat as every notebook here: both target formulas in the seed data are functions of `CustomerId`, not real underwriting outcomes.

In [1]:
from datetime import date

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

from db_utils import bulk_insert, get_connection

FEATURE_COLS = [
    "Age", "Gender", "Region", "CustomerType", "Segment", "CustomerStatus",
    "Balance", "AccountType",
]
TODAY = date.today()

conn = get_connection()
seed = pd.read_sql(
    """
    SELECT c.CustomerId, c.Age, c.Gender, c.Region, c.CustomerType, c.Segment, c.CustomerStatus,
           a.Balance, a.AccountType,
           l.LendingModelScore, l.Eligible, l.ApprovedLimit, l.DefaultRiskScore
    FROM dbo.DimCustomer c
    JOIN dbo.FactCustomerAccount a ON a.CustomerId = c.CustomerId
    JOIN dbo.FactCustomerLending l ON l.CustomerId = c.CustomerId
    WHERE c.CustomerId <= 100000;
    """,
    conn,
)
conn.close()

new_customers = pd.read_csv("data/new_customers_features.csv")
print("Seed:", seed.shape, " New:", new_customers.shape)

/var/folders/xy/sts9z3rj3w3_957f1n6sltrm0000gn/T/ipykernel_39819/733962396.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  seed = pd.read_sql(


Seed: (100000, 13)  New: (100000, 15)


## Stage 1 — Eligibility (`LendingModelScore` → `Eligible`)

In [2]:
combined = pd.concat([seed[FEATURE_COLS], new_customers[FEATURE_COLS]], keys=["seed", "new"])
combined_encoded = pd.get_dummies(combined, drop_first=True)

X_seed = combined_encoded.loc["seed"].reset_index(drop=True)
X_new = combined_encoded.loc["new"].reset_index(drop=True)
y_seed_score = seed["LendingModelScore"].reset_index(drop=True)

X_train, X_test, y_train, y_test = train_test_split(X_seed, y_seed_score, test_size=0.2, random_state=42)
eval_model = RandomForestRegressor(n_estimators=300, max_depth=10, random_state=42)
eval_model.fit(X_train, y_train)
y_pred = eval_model.predict(X_test)
print("Stage 1 (LendingModelScore) - MAE:", round(mean_absolute_error(y_test, y_pred), 4),
      " R2:", round(r2_score(y_test, y_pred), 4))

lending_score_model = RandomForestRegressor(n_estimators=300, max_depth=10, random_state=42)
lending_score_model.fit(X_seed, y_seed_score)
predicted_lending_score = np.clip(lending_score_model.predict(X_new), 0, 1).round(4)
predicted_eligible = (predicted_lending_score > 0.55).astype(int)

print("Eligible rate:", predicted_eligible.mean().round(4))

Stage 1 (LendingModelScore) - MAE: 0.1742  R2: 0.0


Eligible rate: 0.9541


In [3]:
# ApprovedLimit isn't modeled - in the seed data it's a formula of CustomerId alone,
# uncorrelated with anything else, so there's no real relationship to learn. Generated
# here tied to account balance instead, which is at least a plausible basis for a limit.
rng = np.random.default_rng(7)
approved_limit = np.round(
    np.clip(100_000 + new_customers["Balance"] * 0.8 + rng.normal(0, 50_000, size=len(new_customers)), 50_000, 3_000_000),
    2,
)
approved_limit.describe() if hasattr(approved_limit, "describe") else pd.Series(approved_limit).describe()

count    1.000000e+05
mean     1.949597e+05
std      7.942143e+04
min      5.000000e+04
25%      1.414594e+05
50%      1.864521e+05
75%      2.371751e+05
max      1.188913e+06
Name: Balance, dtype: float64

## Stage 2 — Default risk (`DefaultRiskScore` → `Defaulted`), using Stage 1 outputs as extra features

In [4]:
X_seed_stage2 = X_seed.copy()
X_seed_stage2["LendingModelScore"] = seed["LendingModelScore"].reset_index(drop=True)
X_seed_stage2["Eligible"] = seed["Eligible"].reset_index(drop=True)
X_seed_stage2["ApprovedLimit"] = seed["ApprovedLimit"].reset_index(drop=True)

X_new_stage2 = X_new.copy()
X_new_stage2["LendingModelScore"] = predicted_lending_score
X_new_stage2["Eligible"] = predicted_eligible
X_new_stage2["ApprovedLimit"] = approved_limit.values

y_seed_default = seed["DefaultRiskScore"].reset_index(drop=True)

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_seed_stage2, y_seed_default, test_size=0.2, random_state=42
)
eval_model2 = RandomForestRegressor(n_estimators=300, max_depth=10, random_state=42)
eval_model2.fit(X_train2, y_train2)
y_pred2 = eval_model2.predict(X_test2)
print("Stage 2 (DefaultRiskScore) - MAE:", round(mean_absolute_error(y_test2, y_pred2), 4),
      " R2:", round(r2_score(y_test2, y_pred2), 4))

Stage 2 (DefaultRiskScore) - MAE: 0.0335  R2: 0.9538


In [5]:
final_model2 = RandomForestRegressor(n_estimators=300, max_depth=10, random_state=42)
final_model2.fit(X_seed_stage2, y_seed_default)

predicted_default_score = np.clip(final_model2.predict(X_new_stage2), 0, 1).round(4)
predicted_defaulted = (predicted_default_score > 0.88).astype(int)

results = pd.DataFrame({
    "CustomerId": new_customers["CustomerId"],
    "LendingModelScore": predicted_lending_score,
    "Eligible": predicted_eligible,
    "ApprovedLimit": approved_limit.values,
    "DefaultRiskScore": predicted_default_score,
    "Defaulted": predicted_defaulted,
    "ModelDate": TODAY,
})

print("Eligible rate:", results["Eligible"].mean().round(4))
print("Default rate:", results["Defaulted"].mean().round(4))
results.head()

Eligible rate: 0.9541
Default rate: 0.0082


,CustomerId,LendingModelScore,Eligible,ApprovedLimit,DefaultRiskScore,Defaulted,ModelDate
0,100001,0.6424,1,176157.27,0.3524,0,2026-07-23
1,100002,0.6563,1,214369.06,0.7839,0,2026-07-23
2,100003,0.6707,1,139459.40,0.3524,0,2026-07-23
3,100004,0.6442,1,240998.21,0.4809,0,2026-07-23
4,100005,0.6495,1,315719.32,0.1218,0,2026-07-23


In [6]:
cols = ["CustomerId", "LendingModelScore", "Eligible", "ApprovedLimit", "DefaultRiskScore", "Defaulted", "ModelDate"]

conn = get_connection()
n = bulk_insert(conn, "dbo.FactCustomerLending", cols, list(results[cols].itertuples(index=False, name=None)))
check = pd.read_sql("SELECT COUNT(*) AS NewLendingRows FROM dbo.FactCustomerLending WHERE CustomerId > 100000;", conn)
conn.close()
print(f"Inserted {n:,} rows into FactCustomerLending")
check

Inserted 100,000 rows into FactCustomerLending


/var/folders/xy/sts9z3rj3w3_957f1n6sltrm0000gn/T/ipykernel_39819/3366599363.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  check = pd.read_sql("SELECT COUNT(*) AS NewLendingRows FROM dbo.FactCustomerLending WHERE CustomerId > 100000;", conn)


,NewLendingRows
0,100000
